# 02 · Leakage-corrected training (headline reanalysis)

Directly addresses **Reviewer 1.10** and **Reviewer 3.5**. The original pipeline split
graphs with `torch.random_split` at the **subgraph** level, letting near-duplicate
subgraphs from the same patient appear in both train and validation → **leakage**.

We train two models with an **identical** architecture/hyperparameters/training budget:
- **(A) honest** — patient-disjoint split
- **(B) leaky** — random subgraph split (reproduces the original)

…and quantify the optimism gap. Runtime: ~5–10 min on CPU for both.

> Note: this is a *reduced* protocol (hop-3, early stopping). The leaky model reproduces
> the published AUC range, so the protocol is representative; the **gap** is the robust
> finding. Definitive honest headline numbers require the full pipeline on GPU.

In [ ]:
# --- notebook preamble: locate the reusable analysis library ------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
LIB = os.path.join(os.path.dirname(os.path.abspath("__file__")), "lib")
if not os.path.isdir(LIB):
    LIB = os.path.join(os.getcwd(), "lib")
sys.path.insert(0, LIB)

# _lib_raman defines ROOT / DATA / MODEL_PATH / META_PATH (absolute paths to the
# original project). If you moved the data, override them here, e.g.:
#   import _lib_raman as L; L.ROOT = "/path/to/Project_Raman"; L.DATA = ...
import _lib_raman as L
print("Project ROOT :", L.ROOT)
print("Graph DATA   :", L.DATA)
print("Model        :", L.MODEL_PATH)
print("Classes      :", L.CLASSES)

import _lib_train as T
import numpy as np, pandas as pd, torch, time

## 1 · Assemble hop-3 subgraphs (median ~26 nodes)

In [ ]:
graphs = T.assemble_graphs(hop=3, graph_set='small')
print(len(graphs), 'subgraphs from', len(set(g.patient_id for g in graphs)), 'patients')
params = torch.load(L.MODEL_PATH, weights_only=False, map_location='cpu')['params']

## 2 · (A) HONEST — patient-disjoint split

In [ ]:
tr, te, test_pats = T.stratified_patient_split(graphs, test_frac=0.30, seed=42)
print(f'train {len(tr)} sg / test {len(te)} sg / {len(test_pats)} held-out patients')
modelA, histA = T.train_model(tr, te, 'cpu', params, epochs=150, patience=18, bias=True)
dfA = T.predict(modelA, te, 'cpu')
patA = T.aggregate_patient(dfA)
print('subgraph :', T.overall_metrics(dfA))
print('patient  :', T.overall_metrics(patA))

In [ ]:
mA_sg = T.per_class_metrics(dfA, 'subgraph')
mA_pt = T.per_class_metrics(patA, 'patient')
print('Per-class AUC (subgraph, honest, bootstrap 95% CI):')
mA_sg[['class','n_pos','auc','auc_lo','auc_hi']].round(3)

## 3 · (B) LEAKY — random subgraph split (reproduces the original)

In [ ]:
trL, teL, _ = T.random_subgraph_split(graphs, test_frac=0.30, seed=42)
modelB, histB = T.train_model(trL, teL, 'cpu', params, epochs=150, patience=18, bias=True)
dfB = T.predict(modelB, teL, 'cpu')
mB_sg = T.per_class_metrics(dfB, 'subgraph')
print('leaky subgraph:', T.overall_metrics(dfB))
mB_sg[['class','n_pos','auc']].round(3)

## 4 · Optimism gap + calibration

In [ ]:
honest_auc = float(np.nanmean(mA_sg['auc'])); leaky_auc = float(np.nanmean(mB_sg['auc']))
print(f'Mean subgraph AUC  — leaky {leaky_auc:.3f}  vs  honest {honest_auc:.3f}  '
      f'(ΔAUC = {leaky_auc-honest_auc:.3f})')
cal_sg, ece_sg = T.calibration(dfA); cal_pt, ece_pt = T.calibration(patA)
print(f'Calibration ECE — subgraph {ece_sg:.3f} | patient {ece_pt:.3f}')
cal_sg

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,3))
ax.bar(['leaky','honest'], [leaky_auc, honest_auc], color=['#D55E00','#0072B2'])
ax.set_ylabel('Mean one-vs-rest AUC'); ax.set_ylim(0,1)
ax.set_title('Effect of the train/val split'); plt.show()